<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

In [ ]:
# TODO 0: 실습을 위해 아래 패키지를 import하고, diamonds 데이터셋을 준비해주세요.
import numpy as np
import seaborn as sns
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

# diamonds 데이터셋 로드
df = sns.load_dataset("diamonds")
numeric_df = df.select_dtypes(include=["number"])

X = numeric_df.drop(columns=["price"]).values
y = df["price"].values

print(f"데이터: diamonds ({len(X)}개)")
print(f"피처: {list(numeric_df.drop(columns=['price']).columns)}")
print(f"타겟: price")

데이터: diamonds (53940개)
피처: ['carat', 'depth', 'table', 'x', 'y', 'z']
타겟: price


</br>

# 학습 내용
>이번 장에서는 <strong>데이터 분할(Data Splitting)</strong>에 대해 학습합니다.</br></br>
>학습/검증 데이터 분리, 셔플링, 그리고 데이터 누수 방지를 학습해봅시다.

</br>

# 데이터 분할의 필요성
> 모델이 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습에 사용하지 않은 새로운 데이터</mark>에서도 잘 동작하는지 평가하기 위해 데이터를 나눕니다.

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">구분</th>
      <th>역할</th>
      <th style="text-align:center">비율 (일반적)</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">Train</td><td>모델 학습에 사용</td><td style="text-align:center">60~80%</td></tr>
    <tr><td style="text-align:center">Validation</td><td>하이퍼파라미터 튜닝, 조기 종료 판단</td><td style="text-align:center">10~20%</td></tr>
    <tr><td style="text-align:center">Test</td><td>최종 성능 평가 (한 번만 사용)</td><td style="text-align:center">10~20%</td></tr>
  </tbody>
</table>

</br>

In [2]:
# TODO 1: X(price 제외)의 전체 데이터 개수를 n에 저장해봅시다.
dataset_count = len(X)

print(dataset_count)


53940


In [3]:
# TODO 2: shuffled_index에 인덱스를 무작위로 섞어 저장합니다.

shuffled_index = np.random.permutation(dataset_count) # 순열 만들기
print(shuffled_index[:5])

[24113 19001 48245 18269 27564]


In [4]:
# TODO 3: X와 y에 무작위로 섞은 인덱스를 적용해봅시다.

X_shuffled = X[shuffled_index]
y_shuffled = y[shuffled_index]

print(X_shuffled)

[[ 2.03 62.7  59.    8.02  8.08  5.05]
 [ 1.   61.4  59.    6.47  6.39  3.95]
 [ 0.69 58.4  61.7   5.75  5.82  3.38]
 ...
 [ 1.01 62.8  59.    6.34  6.37  3.99]
 [ 0.74 65.4  53.    5.73  5.71  3.74]
 [ 0.52 61.   57.    5.18  5.21  3.17]]


💡permutation vs shuffle
> `np.random.permutation(n)`은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">새로운 배열을 반환</mark>합니다.</br>
> `np.random.shuffle(arr)`은 원본 배열을 직접 수정합니다.</br>
> 인덱스 배열을 만들어 X와 y에 동시 적용하려면 `permutation`이 편리합니다.

</br>

# Train/Validation 분할 구현

In [5]:
# TODO 4: train_ratio에 0.8 (학습 비율)을 저장하고, cut에 분할 지점을 저장해봅시다.

train_ratio = 0.8 # 80% 를 의미

cut_index = int(train_ratio * dataset_count)

print(cut_index)

43152


In [6]:
# TODO 5: 셔플된 인덱스를 분할 기준으로 학습용과 검증용 인덱스로 나눠 X_train, X_valid, y_train, y_valid를 만들어봅시다.

train_index = shuffled_index[:cut_index]
vaild_index = shuffled_index[cut_index:]

X_train, X_vaild = X[train_index], X[vaild_index]
y_train, y_vaild = y[train_index], y[vaild_index]

print(len(X_train), len(X_vaild))

43152 10788


</br>

# 표준화 안전장치
> 표준화 시 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">학습 데이터의 통계값만</mark> 사용하여 학습/검증 데이터를 변환합니다.</br>
> 표준편차가 0인 열이 존재하면 나눗셈 에러가 발생하므로, 이를 확인하고 안전하게 처리합니다.

In [ ]:
# TODO 6: X_train의 평균과 표준편차를 구해봅시다.
mu = np.mean(X_train, axis=0) # 평균

sigma = np.std(X_train, axis=0) #표준편차

print(mu, sigma)

[ 0.7975394  61.74158788 57.45331618  5.73055548  5.73380075  3.53798341] [0.47358072 1.42988625 2.23281424 1.12124947 1.14171712 0.70815481]


In [9]:
# TODO 7: 표준편차가 0인 열이 있는지 확인해봅시다.

zero_mask = (sigma==0)
print(zero_mask)

[False False False False False False]


In [10]:
# TODO 8: 표준편차가 0인 열을 1.0으로 대체해봅시다.

sigma = np.where(sigma==0, 1.0, sigma)
print(sigma)

[0.47358072 1.42988625 2.23281424 1.12124947 1.14171712 0.70815481]


In [11]:
# TODO 9: 대체 후 표준편차가 0인 열이 없는지 다시 확인해봅시다.

zero_mask = (sigma==0)
print(zero_mask)

[False False False False False False]


In [13]:
# TODO 10: X_train과 X_valid를 표준화해봅시다.

X_train = (X_train-mu)/sigma
X_vaild = (X_vaild-mu)/sigma

print(np.mean(X_train, axis=0))
print(np.std(X_train, axis=0))

[ -1.68406222 -43.17937043 -25.73134618  -5.11086574  -5.02208529
  -4.99605928]
[2.11157245 0.69935633 0.44786529 0.89186219 0.8758737  1.41212061]


💡표준편차가 0인 경우
> 모든 값이 동일한 열은 표준편차가 0이 되어 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">나눗셈 에러(ZeroDivisionError)</mark>가 발생합니다.</br>
> `np.where(sigma == 0, 1.0, sigma)`로 0을 1.0으로 대체하면, 해당 열은 `(x - mu) / 1.0 = 0`으로 변환되어 안전합니다.</br>
> 대체 후 반드시 0인 열이 남아있지 않은지 재확인하는 습관을 들여야합니다.

</br>

# 참고: 데이터 누수 (Data Leakage)
> 검증/테스트 데이터의 정보가 학습 과정에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">유입되는 것</mark>을 데이터 누수라고 합니다.

## 대표적인 누수 사례

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">유형</th>
      <th>설명</th>
      <th>방지 방법</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">전체 데이터 표준화</td><td>전체 데이터로 평균/표준편차를 구한 뒤 표준화</td><td>학습 데이터의 통계값만 사용</td></tr>
    <tr><td style="text-align:center">분할 전 피처 엔지니어링</td><td>전체 데이터로 파생 변수 생성 후 분할</td><td>분할 후 학습 데이터만으로 변환</td></tr>
    <tr><td style="text-align:center">미래 정보 포함</td><td>시계열에서 미래 시점 데이터를 피처로 사용</td><td>시간 기반 분할 적용</td></tr>
  </tbody>
</table>

> 데이터 누수가 발생하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">실제보다 높은 성능</mark>이 나와 모델을 과신하게 됩니다.</br>
> 실무에서는 항상 **분할 → 학습 데이터 기준 변환** 순서를 지킵니다.